# Clinical Trials RAG Pipeline
## MSc Generative AI — University of Warwick

In [1]:
import os
from pathlib import Path

# Fix working directory to project root
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

print(f"Working directory: {Path.cwd()}")

Working directory: /Users/sohail/Documents/clinical-trials-rag


## 1. Data Ingestion

In [2]:
import sys
sys.path.insert(0, "..")

import logging
logging.basicConfig(level=logging.INFO)

from src.ingest import ingest_trials, fetch_trials_for_condition

/Users/sohail/Documents/clinical-trials-rag/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
# Quick smoke test: fetch a small batch for one condition
test_trials = fetch_trials_for_condition("Glioblastoma", page_size=5)
print(f"Fetched {len(test_trials)} trial(s)")

if test_trials:
    t = test_trials[0]
    print(f"  NCT ID:    {t['nct_id']}")
    print(f"  Title:     {t['brief_title'][:80]}")
    print(f"  Status:    {t['overall_status']}")
    print(f"  Phase:     {t['phase']}")
    print(f"  Conditions: {t['conditions']}")
    print(f"  Fields:    {list(t.keys())}")

INFO:src.ingest:Fetching Glioblastoma - page 1
INFO:src.ingest:Fetching Glioblastoma - page 2
INFO:src.ingest:Fetching Glioblastoma - page 3
INFO:src.ingest:Fetching Glioblastoma - page 4
INFO:src.ingest:Fetching Glioblastoma - page 5
INFO:src.ingest:Fetching Glioblastoma - page 6
INFO:src.ingest:Fetching Glioblastoma - page 7
INFO:src.ingest:Fetching Glioblastoma - page 8
INFO:src.ingest:Fetching Glioblastoma - page 9
INFO:src.ingest:Fetching Glioblastoma - page 10
INFO:src.ingest:Fetching Glioblastoma - page 11
INFO:src.ingest:Fetching Glioblastoma - page 12
INFO:src.ingest:Fetching Glioblastoma - page 13
INFO:src.ingest:Fetching Glioblastoma - page 14
INFO:src.ingest:Fetching Glioblastoma - page 15
INFO:src.ingest:Fetching Glioblastoma - page 16
INFO:src.ingest:Fetching Glioblastoma - page 17
INFO:src.ingest:Fetching Glioblastoma - page 18
INFO:src.ingest:Fetching Glioblastoma - page 19
INFO:src.ingest:Fetching Glioblastoma - page 20
INFO:src.ingest:Fetching Glioblastoma - page 21
I

Fetched 1567 trial(s)
  NCT ID:    NCT02852655
  Title:     A Pilot Surgical Trial To Evaluate Early Immunologic Pharmacodynamic Parameters 
  Status:    ACTIVE_NOT_RECRUITING
  Phase:     PHASE1
  Conditions: ['Brain Cancer']
  Fields:    ['nct_id', 'brief_title', 'official_title', 'brief_summary', 'detailed_description', 'eligibility_criteria', 'intervention_names', 'intervention_types', 'phase', 'overall_status', 'lead_sponsor_name', 'start_date', 'primary_completion_date', 'last_update_post_date', 'conditions', 'source_conditions', 'study_type']


In [4]:
# Full ingestion (all 4 conditions) — takes a few minutes
# Uncomment when ready:
# trials = ingest_trials()
# print(f"Total unique trials: {len(trials)}")

## 2. Chunking

In [5]:
import sys
sys.path.insert(0, "..")

from src.chunking import chunk_trial, run_chunking_pipeline, load_raw_trials
from collections import Counter

# Quick test: chunk a single trial
trials = load_raw_trials()
sample = trials[0]
sample_chunks = chunk_trial(sample)

print(f"Trial {sample['nct_id']} produced {len(sample_chunks)} chunks:")
for c in sample_chunks:
    print(f"  {c['chunk_type']:15s} | {len(c['text']):5d} chars | has source_conditions: {'source_conditions' in c}")
    print(f"    preview: {c['text'][:80]}...")
print()

# Verify parent metadata is on every chunk
required_meta = ["nct_id", "conditions", "source_conditions", "phase", "overall_status"]
for c in sample_chunks:
    missing = [k for k in required_meta if k not in c]
    assert not missing, f"Chunk {c['chunk_id']} missing metadata: {missing}"
print("✓ All chunks carry required parent metadata")

INFO:src.chunking:Loading trials from /Users/sohail/Documents/clinical-trials-rag/data/raw/trials_full_20260330_232310.json


Trial NCT07031765 produced 5 chunks:
  metadata        |   627 chars | has source_conditions: True
    preview: NCT ID: NCT07031765
Title: Peds CHAMP1ON - Hematopoietic Stem Cell And Monoclona...
  summary         |   642 chars | has source_conditions: True
    preview: Official Title: Peds CHAMP1ON - Combinatorial Hematopoietic Stem Cell And Monocl...
  eligibility     |  7216 chars | has source_conditions: True
    preview: Eligibility Criteria for Peds CHAMP1ON - Hematopoietic Stem Cell And Monoclonal ...
  intervention    |   218 chars | has source_conditions: True
    preview: Interventions for Peds CHAMP1ON - Hematopoietic Stem Cell And Monoclonal Antibod...
  description     |   895 chars | has source_conditions: True
    preview: Detailed Description for Peds CHAMP1ON - Hematopoietic Stem Cell And Monoclonal ...

✓ All chunks carry required parent metadata


In [6]:
# Full chunking pipeline — run this after confirming smoke test above
chunks = run_chunking_pipeline()
print(f"\nTotal chunks: {len(chunks)}")

# Distribution by chunk type
type_counts = Counter(c["chunk_type"] for c in chunks)
print("\nChunk type distribution:")
for ctype, count in type_counts.most_common():
    print(f"  {ctype}: {count}")

# Distribution by source_condition (checks for SCLC imbalance awareness)
cond_counts = Counter()
for c in chunks:
    for sc in c["source_conditions"].split(", "):
        if sc:
            cond_counts[sc] += 1

print("\nChunks per source condition:")
for cond, count in cond_counts.most_common():
    pct = 100 * count / len(chunks)
    print(f"  {cond}: {count} ({pct:.1f}%)")

INFO:src.chunking:Loading trials from /Users/sohail/Documents/clinical-trials-rag/data/raw/trials_full_20260330_232310.json
INFO:src.chunking:Chunking complete: 63906 chunks from 13967 trials (0 trials produced no chunks)
INFO:src.chunking:  metadata: 13967 chunks
INFO:src.chunking:  summary: 13966 chunks
INFO:src.chunking:  eligibility: 13966 chunks
INFO:src.chunking:  intervention: 12964 chunks
INFO:src.chunking:  description: 9043 chunks
INFO:src.chunking:Saved 63906 chunks to /Users/sohail/Documents/clinical-trials-rag/data/processed/chunks.json



Total chunks: 63906

Chunk type distribution:
  metadata: 13967
  summary: 13966
  eligibility: 13966
  intervention: 12964
  description: 9043

Chunks per source condition:
  small cell lung cancer: 28520 (44.6%)
  ovarian cancer: 16786 (26.3%)
  pancreatic cancer: 15439 (24.2%)
  glioblastoma: 7378 (11.5%)


## 3. Retrieval (Hybrid BM25 + Dense)

In [7]:
import sys
sys.path.insert(0, "..")

from src.retrieval import (
    load_chunks, build_indexes, hybrid_search, parse_query_filters
)

# Load chunks (same list for both BM25 and dense — critical requirement)
chunks = load_chunks()
print(f"Loaded {len(chunks)} chunks")

# Build both indexes (ChromaDB upsert + BM25) — takes a few minutes first time
collection, bm25_index = build_indexes(chunks)
print(f"ChromaDB collection: {collection.count()} documents")
print(f"BM25 index: {len(bm25_index.chunks)} documents")
assert collection.count() == len(bm25_index.chunks), "BM25 and dense must share same corpus!"
print("✓ Both indexes built on same corpus")

INFO:src.retrieval:Loaded 63906 chunks from /Users/sohail/Documents/clinical-trials-rag/data/processed/chunks.json
/Users/sohail/Documents/clinical-trials-rag/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 63906 chunks


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:src.retrieval:ChromaDB collection already contains 63906 documents, skipping upsert
INFO:src.retrieval:BM25 index built with 63906 documents


ChromaDB collection: 63906 documents
BM25 index: 63906 documents
✓ Both indexes built on same corpus


In [8]:
# Test query filter parsing
test_queries = [
    "What phase 3 glioblastoma trials are recruiting?",
    "Show me completed ovarian cancer studies",
    "SCLC immunotherapy trials",
    "Tell me about pancreatic cancer phase 2 treatments",
]

for q in test_queries:
    filters = parse_query_filters(q)
    print(f"Query: {q}")
    print(f"  Parsed: {filters}\n")

Query: What phase 3 glioblastoma trials are recruiting?
  Parsed: {'condition': 'glioblastoma', 'phase': 'PHASE3', 'status': 'RECRUITING'}

Query: Show me completed ovarian cancer studies
  Parsed: {'condition': 'ovarian cancer', 'status': 'COMPLETED'}

Query: SCLC immunotherapy trials
  Parsed: {'condition': 'small cell lung cancer'}

Query: Tell me about pancreatic cancer phase 2 treatments
  Parsed: {'condition': 'pancreatic cancer', 'phase': 'PHASE2'}



In [9]:
# Test hybrid search
query = "What phase 3 glioblastoma trials are currently recruiting?"
results = hybrid_search(query, collection, bm25_index, top_k=20)

print(f"Query: {query}")
print(f"Results: {len(results)}\n")
print("Top 5:")
for r in results[:5]:
    print(f"  #{r['rrf_rank']} | {r['chunk_id'][:40]:40s} | RRF: {r['rrf_score']:.6f} | {r['chunk_type']}")
    print(f"         {r['text'][:100]}...")
    print()

INFO:src.retrieval:Parsed query filters: {'condition': 'glioblastoma', 'phase': 'PHASE3', 'status': 'RECRUITING'}
Batches: 100%|██████████| 1/1 [00:00<00:00,  9.35it/s]
INFO:src.retrieval:Hybrid search: 20 BM25 + 20 dense → 20 fused results


Query: What phase 3 glioblastoma trials are currently recruiting?
Results: 20

Top 5:
  #1 | NCT03970447_metadata                     | RRF: 0.009836 | metadata
         NCT ID: NCT03970447
Title: A Trial to Evaluate Multiple Regimens in Newly Diagnosed and Recurrent Gl...

  #2 | NCT03970447_description                  | RRF: 0.009677 | description
         Detailed Description for A Trial to Evaluate Multiple Regimens in Newly Diagnosed and Recurrent Glio...

  #3 | NCT03970447_intervention                 | RRF: 0.009524 | intervention
         Interventions for A Trial to Evaluate Multiple Regimens in Newly Diagnosed and Recurrent Glioblastom...

  #4 | NCT05271240_metadata                     | RRF: 0.009375 | metadata
         NCT ID: NCT05271240
Title: Repeated Superselective Intraarterial Cerebral Infusion (SIACI) of Bevaci...

  #5 | NCT07195591_metadata                     | RRF: 0.009231 | metadata
         NCT ID: NCT07195591
Title: Beginning Radiation Immediately With Gam

## 4. Cross-Encoder Re-ranking

In [10]:
from src.retrieval import load_chunks, build_indexes, hybrid_search
from src.reranker import rerank

# Ensure indexes are available (instant if already built)
if 'collection' not in dir() or 'bm25_index' not in dir():
    chunks = load_chunks()
    collection, bm25_index = build_indexes(chunks)

# Rerank the top 20 hybrid results using the ORIGINAL query (not HyDE)
query = "What phase 3 glioblastoma trials are currently recruiting?"
hybrid_results = hybrid_search(query, collection, bm25_index, top_k=20)

reranked = rerank(query=query, candidates=hybrid_results, top_k=5)

print(f"Query: {query}")
print(f"Reranked: {len(hybrid_results)} → {len(reranked)}\n")
print("Top 5 after reranking:")
for r in reranked:
    print(f"  #{r['rerank_rank']} | {r['chunk_id'][:40]:40s} | rerank: {r['rerank_score']:.4f} | RRF: {r['rrf_score']:.6f} | {r['chunk_type']}")
    print(f"         {r['text'][:100]}...")
    print()

# Show how reranking changed the ordering
print("--- Ordering comparison ---")
print("Before (RRF top 5):")
for r in hybrid_results[:5]:
    print(f"  {r['chunk_id'][:45]}")
print("\nAfter (reranked top 5):")
for r in reranked:
    print(f"  {r['chunk_id'][:45]}")

INFO:src.retrieval:Parsed query filters: {'condition': 'glioblastoma', 'phase': 'PHASE3', 'status': 'RECRUITING'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 116.90it/s]
INFO:src.retrieval:Hybrid search: 20 BM25 + 20 dense → 20 fused results
INFO:src.reranker:Loading cross-encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: mps
INFO:src.reranker:Cross-encoder model loaded
Batches: 100%|██████████| 1/1 [00:00<00:00,  4.86it/s]
INFO:src.reranker:Reranked 20 candidates → top 5 (score range: 3.7186 to 5.0176)


Query: What phase 3 glioblastoma trials are currently recruiting?
Reranked: 20 → 5

Top 5 after reranking:
  #1 | NCT03548571_metadata                     | rerank: 5.0176 | RRF: 0.008333 | metadata
         NCT ID: NCT03548571
Title: Dendritic Cell Immunotherapy Against Cancer Stem Cells in Glioblastoma Pa...

  #2 | NCT03970447_metadata                     | rerank: 4.8634 | RRF: 0.009836 | metadata
         NCT ID: NCT03970447
Title: A Trial to Evaluate Multiple Regimens in Newly Diagnosed and Recurrent Gl...

  #3 | NCT05669820_metadata                     | rerank: 4.7683 | RRF: 0.008955 | metadata
         NCT ID: NCT05669820
Title: Antisecretory Factor Glioblastoma Phase 2
Official Title: Antisecretory F...

  #4 | NCT06556563_metadata                     | rerank: 4.6557 | RRF: 0.008696 | metadata
         NCT ID: NCT06556563
Title: EF-41/KEYNOTE D58: Phase 3 Study of Optune Concomitant With Temozolomide ...

  #5 | NCT05904119_metadata                     | rerank: 3.7186 | RR

## 5. HyDE (Hypothetical Document Embeddings)

In [11]:
from src.retrieval import load_chunks, build_indexes, hybrid_search
from src.reranker import rerank
from src.hyde import hyde_search, is_nct_id_lookup

# Ensure indexes are available
if 'collection' not in dir() or 'bm25_index' not in dir():
    chunks = load_chunks()
    collection, bm25_index = build_indexes(chunks)

# Test 1: Semantic query — HyDE should activate
query = "What immunotherapy trials are available for glioblastoma patients who failed temozolomide?"
results, hyde_doc = hyde_search(query, collection, bm25_index, hybrid_search, top_k=20)

print(f"Query: {query}")
print(f"HyDE activated: {hyde_doc is not None}")
print(f"Results: {len(results)}\n")

if hyde_doc:
    print("=== HyDE Generated Document ===")
    print(hyde_doc[:500])
    print("...\n")

# Rerank with ORIGINAL query (not HyDE doc)
reranked = rerank(query=query, candidates=results, top_k=5)
print("Top 5 after reranking:")
for r in reranked:
    print(f"  #{r['rerank_rank']} | {r['chunk_id'][:40]:40s} | rerank: {r['rerank_score']:.4f}")
    print(f"         {r['text'][:100]}...")
    print()

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.hyde:HyDE document generated (989 chars)
INFO:src.hyde:HyDE search with filters: {'condition': 'glioblastoma'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 35.71it/s]
INFO:src.hyde:HyDE search: 20 BM25 + 20 dense → 20 fused results


Query: What immunotherapy trials are available for glioblastoma patients who failed temozolomide?
HyDE activated: True
Results: 20

=== HyDE Generated Document ===
- Title: A Phase 2 Study of Pembrolizumab in Patients with Recurrent Glioblastoma Following Temozolomide Treatment
- Condition: Glioblastoma Multiforme
- Phase: Phase 2
- Status: Recruiting
- Intervention: Pembrolizumab (Keytruda)
- Eligibility Summary: Patients aged 18 years and older with a confirmed diagnosis of recurrent glioblastoma who have previously failed temozolomide therapy are eligible. Exclusion criteria include prior treatment with anti-PD-1 or anti-PD-L1 therapies and significant 
...



Batches: 100%|██████████| 1/1 [00:00<00:00, 17.33it/s]
INFO:src.reranker:Reranked 20 candidates → top 5 (score range: 0.8326 to 2.3513)


Top 5 after reranking:
  #1 | NCT04977375_summary                      | rerank: 2.3513
         Official Title: Phase Ib/II Trial of Anti-PD-1 Immunotherapy and Stereotactic Radiation in Patients ...

  #2 | NCT02852655_summary                      | rerank: 2.2271
         Official Title: A Pilot Surgical Trial To Evaluate Early Immunologic Pharmacodynamic Parameters For ...

  #3 | NCT03665545_description                  | rerank: 1.7853
         Detailed Description for Pembrolizumab in Association With the IMA950/Poly-ICLC for Relapsing Gliobl...

  #4 | NCT02337686_summary                      | rerank: 1.0917
         Official Title: Pharmacodynamic Study of Pembrolizumab in Patients With Recurrent Glioblastoma

Summ...

  #5 | NCT03665545_summary                      | rerank: 0.8326
         Official Title: Pembrolizumab in Association With the Multipeptide Vaccine IMA950 Adjuvanted With Po...



In [12]:
# Test 2: NCT ID lookup — HyDE should be skipped
nct_query = "Tell me about NCT03970447"
nct_results, nct_hyde_doc = hyde_search(nct_query, collection, bm25_index, hybrid_search, top_k=20)

print(f"Query: {nct_query}")
print(f"HyDE activated: {nct_hyde_doc is not None}")
print(f"Is NCT ID lookup: {is_nct_id_lookup(nct_query)}")
print(f"Results: {len(nct_results)}")
assert nct_hyde_doc is None, "HyDE should NOT activate for NCT ID lookups!"
print("✓ HyDE correctly skipped for NCT ID lookup")

INFO:src.hyde:NCT ID detected in query — skipping HyDE
INFO:src.retrieval:NCT ID lookup: found 5 chunks for NCT03970447


Query: Tell me about NCT03970447
HyDE activated: False
Is NCT ID lookup: True
Results: 5
✓ HyDE correctly skipped for NCT ID lookup


## 6. Generation (Baseline vs RAG + Prompt Engineering)

In [13]:
from src.retrieval import load_chunks, build_indexes, hybrid_search
from src.reranker import rerank
from src.hyde import hyde_search
from src.generation import generate_baseline, generate_rag, format_context, run_full_pipeline

# Ensure indexes are available
if 'collection' not in dir() or 'bm25_index' not in dir():
    chunks = load_chunks()
    collection, bm25_index = build_indexes(chunks)

# Test query
query = "What immunotherapy trials are recruiting for glioblastoma?"

# --- Baseline (LLM only, no context) ---
print("=" * 80)
print("BASELINE (no retrieval)")
print("=" * 80)
baseline = generate_baseline(query)
print(baseline[:600])
print("...\n" if len(baseline) > 600 else "\n")

BASELINE (no retrieval)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.generation:Baseline generation complete (1696 chars)


As of my last update in October 2023, there are several immunotherapy trials for glioblastoma that may be recruiting participants. However, the specific trials can change frequently, so it's essential to check clinical trial registries or databases for the most current information. Here are some general types of immunotherapy approaches that are often explored in glioblastoma trials:

1. **Checkpoint Inhibitors**: Trials investigating drugs that block proteins like PD-1 or CTLA-4 to enhance the immune response against tumor cells.

2. **CAR T-cell Therapy**: Trials focusing on genetically modi
...



In [14]:
# --- RAG: Prompt A vs B vs C comparison ---
# Retrieve + rerank first (shared across all prompt versions)
candidates, hyde_doc = hyde_search(query, collection, bm25_index, hybrid_search, top_k=20)
reranked = rerank(query=query, candidates=candidates, top_k=5)

print(f"Retrieved {len(candidates)} candidates → reranked to {len(reranked)}")
if hyde_doc:
    print(f"\nHyDE document generated ({len(hyde_doc)} chars)")
print()

for version in ["A", "B", "C"]:
    print("=" * 80)
    print(f"RAG — PROMPT VERSION {version}")
    print("=" * 80)
    answer = generate_rag(query, reranked, prompt_version=version)
    print(answer[:800])
    print("...\n" if len(answer) > 800 else "\n")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.hyde:HyDE document generated (981 chars)
INFO:src.hyde:HyDE search with filters: {'condition': 'glioblastoma', 'status': 'RECRUITING'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 37.17it/s]
INFO:src.hyde:HyDE search: 20 BM25 + 20 dense → 20 fused results
Batches: 100%|██████████| 1/1 [00:00<00:00, 21.39it/s]
INFO:src.reranker:Reranked 20 candidates → top 5 (score range: 1.2980 to 3.5833)


Retrieved 20 candidates → reranked to 5

HyDE document generated (981 chars)

RAG — PROMPT VERSION A


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.generation:RAG generation (prompt A) complete (1969 chars)


Here are some immunotherapy trials currently recruiting for glioblastoma:

1. **Trial of Anti-PD-1 Immunotherapy and Stereotactic Radiation in Patients With Recurrent Glioblastoma**
   - **NCT ID**: NCT04977375
   - **Interventions**: Pembrolizumab (anti-PD-1 drug), Stereotactic Radiation Therapy, Surgical Resection.
   - **Purpose**: To assess the safety, tolerability, and feasibility of pembrolizumab and radiation therapy before surgical resection in patients with recurrent glioblastoma. The trial will also evaluate overall survival and progression-free survival, as well as T cell responses.
   - **Eligibility**: Adults aged 18 and older with confirmed recurrent glioblastoma, tumor size less than 6 cm, and adequate performance status.

2. **T Cells and Pembrolizumab for Recurrent and New
...

RAG — PROMPT VERSION B


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.generation:RAG generation (prompt B) complete (646 chars)


The following immunotherapy trials are recruiting for glioblastoma:

1. **NCT ID: NCT04977375** - This trial assesses the safety and tolerability of pembrolizumab and stereotactic radiation therapy before surgical resection in patients with recurrent glioblastoma.

2. **NCT ID: NCT06157541** - This trial involves allogeneic cytomegalovirus (CMV)-specific T cells as monotherapy and in combination with pembrolizumab for participants with recurrent and newly diagnosed glioblastoma.

3. **NCT ID: NCT05465954** - This trial tests the safety and side effects of efineptakin alfa and pembrolizumab in treating patients with recurrent glioblastoma.


RAG — PROMPT VERSION C


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.generation:RAG generation (prompt C) complete (2619 chars)


Based on the provided clinical trial context, the following immunotherapy trials are currently recruiting for glioblastoma:

### 1. NCT ID: NCT04977375
- **Type**: Intervention
- **Phase**: Ib/II
- **Status**: Recruiting
- **Interventions**:
  - Pembrolizumab (DRUG)
  - Stereotactic Radiation Therapy (RADIATION)
  - Surgical Resection (PROCEDURE)
- **Summary**: This trial assesses the safety, tolerability, and feasibility of pembrolizumab and radiation therapy before surgical resection in patients with recurrent glioblastoma. It aims to evaluate treatment-related adverse events, overall survival, progression-free survival, T cell clonality, CD8 T cell activation, and Tumor Infiltrating Lymphocyte (TIL) score after treatment.
- **Eligibility**:
  - **Inclusion Criteria**: Age 18 years or ol
...



In [15]:
# --- Full pipeline convenience function ---
result = run_full_pipeline(
    query="What phase 2 pancreatic cancer trials are testing combination therapies?",
    collection=collection,
    bm25_index=bm25_index,
    hybrid_search_fn=hybrid_search,
    rerank_fn=rerank,
    hyde_search_fn=hyde_search,
    prompt_version="C",
    use_hyde=True,
)

print(f"Query: {result['query']}")
print(f"Prompt version: {result['prompt_version']}")
print(f"Candidates: {result['num_candidates']} → Reranked: {result['num_reranked']}")
print(f"HyDE used: {result['hyde_document'] is not None}")
print()
print("=" * 80)
print("BASELINE ANSWER")
print("=" * 80)
print(result["baseline_answer"][:500])
print()
print("=" * 80)
print("RAG ANSWER (Prompt C)")
print("=" * 80)
print(result["rag_answer"][:800])

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.hyde:HyDE document generated (871 chars)
INFO:src.hyde:HyDE search with filters: {'condition': 'pancreatic cancer', 'phase': 'PHASE2'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 41.05it/s]
INFO:src.hyde:HyDE search: 20 BM25 + 20 dense → 20 fused results
Batches: 100%|██████████| 1/1 [00:00<00:00, 16.61it/s]
INFO:src.reranker:Reranked 20 candidates → top 5 (score range: 3.9574 to 6.0362)
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.generation:Baseline generation complete (1431 chars)
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.generation:RAG generation (prompt C) complete (2379 chars)


Query: What phase 2 pancreatic cancer trials are testing combination therapies?
Prompt version: C
Candidates: 20 → Reranked: 5
HyDE used: True

BASELINE ANSWER
As of my last update in October 2023, there are several phase 2 clinical trials for pancreatic cancer that are testing combination therapies. These trials often explore the efficacy of combining chemotherapy, targeted therapies, immunotherapy, and other treatment modalities to improve outcomes for patients with pancreatic cancer.

To find specific trials, you can search clinical trial registries such as:

1. **ClinicalTrials.gov**: This is a comprehensive database of privately and publicly funde

RAG ANSWER (Prompt C)
Here are the Phase II clinical trials for pancreatic cancer that are testing combination therapies:

### NCT ID: NCT00202800
- **Title**: Phase II Trial to Evaluate Gemcitabine and Etoposide for Locally Advanced or Metastatic Pancreatic Cancer
- **Status**: Not specified
- **Interventions**: Gemcitabine and Etoposi

## 7. Evaluation & Ablation Study

In [16]:
from src.retrieval import load_chunks, build_indexes, hybrid_search, dense_search
from src.reranker import rerank
from src.hyde import hyde_search
from src.generation import generate_baseline, generate_rag
from src.evaluate import load_test_queries, print_ablation_summary

# Ensure indexes are available
if 'collection' not in dir() or 'bm25_index' not in dir():
    chunks = load_chunks()
    collection, bm25_index = build_indexes(chunks)

# Load test queries
test_queries = load_test_queries()
print(f"Loaded {len(test_queries)} test queries")
for tq in test_queries[:5]:
    print(f"  [{tq['id']}] ({tq['type']:12s}) {tq['query'][:70]}")
print("  ...")

INFO:src.evaluate:Loaded 20 test queries


Loaded 20 test queries
  [Q01] (factual     ) What is the official title and status of trial NCT03970447?
  [Q02] (factual     ) What interventions are used in trial NCT02685605?
  [Q03] (filtering   ) What phase 3 glioblastoma trials are currently recruiting?
  [Q04] (filtering   ) Show me completed phase 2 ovarian cancer trials
  [Q05] (filtering   ) List all recruiting pancreatic cancer trials
  ...


In [17]:
# --- Quick single-query test before running full ablation ---
from src.evaluate import (
    run_baseline, run_naive_rag, run_hybrid_rag,
    run_hybrid_rerank, run_full_system
)

test_q = "What is the official title and status of trial NCT03970447?"

print(f"Query: {test_q}\n")

# Config 1: Baseline
r1 = run_baseline(test_q, generate_baseline)
print(f"[1] Baseline LLM:\n{r1['answer'][:200]}\n")

# Config 5: Full system
r5 = run_full_system(
    test_q, collection, bm25_index,
    generate_rag, hybrid_search, rerank, hyde_search
)
print(f"[5] Full System:\n{r5['answer'][:400]}\n")
print(f"HyDE used: {r5['hyde_document'] is not None}")
print(f"Retrieved NCT IDs: {[c.get('nct_id') for c in r5['retrieved_chunks']]}")

Query: What is the official title and status of trial NCT03970447?



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.generation:Baseline generation complete (252 chars)
INFO:src.hyde:NCT ID detected in query — skipping HyDE
INFO:src.retrieval:NCT ID lookup: found 5 chunks for NCT03970447


[1] Baseline LLM:
The official title of trial NCT03970447 is "A Study to Evaluate the Safety and Efficacy of Tislelizumab in Combination With Chemotherapy in Patients With Advanced Esophageal Squamous Cell Carcinoma." 



Batches: 100%|██████████| 1/1 [00:00<00:00, 15.38it/s]
INFO:src.reranker:Reranked 5 candidates → top 5 (score range: -9.6026 to 7.3966)
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.generation:RAG generation (prompt C) complete (307 chars)


[5] Full System:
The official title of trial NCT03970447 is "GBM AGILE: Global Adaptive Trial Master Protocol: An International, Seamless Phase II/III Response Adaptive Randomization Platform Trial Designed To Evaluate Multiple Regimens In Newly Diagnosed and Recurrent GBM." The current status of the trial is "RECRUITING."

HyDE used: False
Retrieved NCT IDs: ['NCT03970447', 'NCT03970447', 'NCT03970447', 'NCT03970447', 'NCT03970447']


In [18]:
# Full ablation study (20 queries × 5 configs = 100 runs)
# WARNING: This takes ~15-20 minutes and uses ~100 OpenAI API calls.

from src.evaluate import run_ablation_study

results = run_ablation_study(
    test_queries=test_queries,
    collection=collection,
    bm25_index=bm25_index,
    generate_baseline_fn=generate_baseline,
    generate_rag_fn=generate_rag,
    dense_search_fn=dense_search,
    hybrid_search_fn=hybrid_search,
    rerank_fn=rerank,
    hyde_search_fn=hyde_search,
    save=True,
)

print_ablation_summary(results)

INFO:src.evaluate:[1/100] Q01 — 1_baseline_llm: What is the official title and status of trial NCT
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.generation:Baseline generation complete (252 chars)
INFO:src.evaluate:[2/100] Q01 — 2_naive_rag: What is the official title and status of trial NCT
Batches: 100%|██████████| 1/1 [00:00<00:00, 14.12it/s]
INFO:src.reranker:Reranked 20 candidates → top 5 (score range: 6.0514 to 6.7395)
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.generation:RAG generation (prompt C) complete (55 chars)
INFO:src.evaluate:[3/100] Q01 — 3_hybrid_rag: What is the official title and status of trial NCT
INFO:src.retrieval:NCT ID lookup: found 5 chunks for NCT03970447
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:src.generation:RAG generation (prompt C) complete (307 chars)
INFO:src.evaluate:[4/100] Q01 — 4_hybrid


ABLATION STUDY SUMMARY
Config                     Queries      P@5      R@5    Avg Len
-----------------------------------------------------------------
1_baseline_llm                  20    0.000     0.000        1202
2_naive_rag                     20    0.025     0.009        1519
3_hybrid_rag                    20    0.125     0.277        1418
4_hybrid_reranker               20    0.175     0.295        1760
5_full_system                   20    0.175     0.295        1865
Note: P@5 and R@5 only computed for queries with ground truth NCT IDs.
Generation quality (accuracy, grounding, hallucination, completeness)
requires manual scoring — see evaluation/ablation_results_*.json
